In [2]:
import sys, os, requests
sys.path.append('./official/DISCoVeR-main/src/')

import scanpy as sc
import anndata as ad

from metrics import kmeans_ari, MINE
from NBVAE_variants import ZINBVAE, ZINBCVAE, ZINBCSVAENA, ZINBCSVAE, ZINBHCSVAENA, ZINBHCSVAE, ZINBDLVAE, ZINBDIVA, ZINBCCVAE
from VAE_trainers import EpochPyroTrainer, AdversarialEpochPyroTrainer, ThresholdPyroTrainer, AdversarialThresholdPyroTrainer
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
import seaborn as sns

from tqdm import trange, tqdm
from umap import UMAP


import torch, pyro
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import pyro.optim as opt

np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)


cmap_trt = LinearSegmentedColormap.from_list("cmap", ["#42378C", "#D9A404"])
cmap_ct = ListedColormap(sns.color_palette('colorblind').as_hex())

In [3]:
from scipy.io import mmread
RNA_counts = mmread('conversion_files/feed_RNA_counts.mtx').transpose().tocsr().astype('float64')
cell_metadata = pd.read_csv('conversion_files/feed_cell_metadata.csv')
cell_names = pd.read_csv('conversion_files/feed_cells.csv')
cell_metadata.index = cell_names['cell']
gene_metadata = pd.read_csv("conversion_files/feed_genes.csv")

In [4]:
anndata = ad.AnnData(X=RNA_counts, obs=cell_metadata, var=gene_metadata)
anndata.layers['raw'] = anndata.X.copy()

C:\Users\anany\miniconda3\envs\ds_env\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [5]:
# Filter
sc.pp.filter_cells(anndata, min_genes=200)
sc.pp.filter_genes(anndata, min_cells=3)

# Normalize and log-transform
sc.pp.normalize_total(anndata)
sc.pp.log1p(anndata)

# Highly variable genes
sc.pp.highly_variable_genes(anndata, n_top_genes=2000)
anndata = anndata[:, anndata.var['highly_variable']]

In [6]:
import torch.utils.data as utils

np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

batch_size=128
x = torch.FloatTensor(anndata.layers['raw'].copy().toarray())
anndata.obs['time'] = anndata.obs['time'].astype('category')
anndata.obs['zone'] = anndata.obs['zone'].astype('float')
y = torch.FloatTensor(anndata.obs['time'].cat.codes.to_numpy().reshape(-1,1).copy())
y_info = torch.FloatTensor(anndata.obs['zone'])

dataset = utils.TensorDataset(x, y, y_info)
train_set, test_set = dataset, dataset
train_set, test_set = utils.TensorDataset(*train_set[:]), utils.TensorDataset(*test_set[:])
train_loader, test_loader  = torch.utils.data.DataLoader(train_set, shuffle=True, batch_size=batch_size),  torch.utils.data.DataLoader(test_set, shuffle=True, batch_size=batch_size)

C:\Users\anany\AppData\Local\Temp\ipykernel_11856\4176040515.py:9: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  anndata.obs['time'] = anndata.obs['time'].astype('category')
C:\Users\anany\AppData\Local\Temp\ipykernel_11856\4176040515.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  y_info = torch.FloatTensor(anndata.obs['zone'])


In [7]:
from pyfonts import load_font

# load font
font = load_font(
   font_url="https://github.com/stevenpetryk/computer-modern/blob/main/src/cmunrm.ttf?raw=true"
)

In [8]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvae_1 = ZINBCSVAE(2000, [1], latent_dim=10, w_dim=10, w_locs = [0.0, 1.0, 2.0, 3.0, 4.0], w_scales = [0.1, 0.1, 0.1, 0.1, 0.1], recon_weight = 20.0, z_kl_weight = 0.2, w_kl_weight = 1.0, adversarial_weight = 10)
csvae_trainer_1 = EpochPyroTrainer(500, csvae_1, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
csvae_trainer_1.train()

Epoch : 1 || Train Loss: 24105.347 || Test Loss: 13226.749
Epoch : 2 || Train Loss: 12854.914 || Test Loss: 12671.988
Epoch : 3 || Train Loss: 12575.764 || Test Loss: 12465.367
Epoch : 4 || Train Loss: 12450.414 || Test Loss: 12371.378
Epoch : 5 || Train Loss: 12360.891 || Test Loss: 12323.486
Epoch : 6 || Train Loss: 12285.606 || Test Loss: 12206.14
Epoch : 7 || Train Loss: 12240.519 || Test Loss: 12175.434
Epoch : 8 || Train Loss: 12198.605 || Test Loss: 12151.07
Epoch : 9 || Train Loss: 12166.032 || Test Loss: 12111.939
Epoch : 10 || Train Loss: 12134.334 || Test Loss: 12111.907
Epoch : 11 || Train Loss: 12111.591 || Test Loss: 12055.536
Epoch : 12 || Train Loss: 12090.391 || Test Loss: 12079.004
Epoch : 13 || Train Loss: 12063.459 || Test Loss: 12012.092
Epoch : 14 || Train Loss: 12051.26 || Test Loss: 12049.836
Epoch : 15 || Train Loss: 12035.683 || Test Loss: 11980.747
Epoch : 16 || Train Loss: 12022.113 || Test Loss: 11965.734
Epoch : 17 || Train Loss: 11993.906 || Test Loss: 11

In [9]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvae_2 = ZINBCSVAE(2000, [1], latent_dim=10, w_dim=10, w_locs = [0.0, 1.0, 2.0, 3.0, 4.0], w_scales = [0.2, 0.2, 0.2, 0.2, 0.2], recon_weight = 20.0, z_kl_weight = 0.2, w_kl_weight = 1.0, adversarial_weight = 10)
csvae_trainer_2 = EpochPyroTrainer(500, csvae_2, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
csvae_trainer_2.train()

Epoch : 1 || Train Loss: 23795.861 || Test Loss: 13317.231
Epoch : 2 || Train Loss: 12864.482 || Test Loss: 12656.127
Epoch : 3 || Train Loss: 12545.644 || Test Loss: 12454.362
Epoch : 4 || Train Loss: 12415.153 || Test Loss: 12353.23
Epoch : 5 || Train Loss: 12327.819 || Test Loss: 12267.645
Epoch : 6 || Train Loss: 12257.714 || Test Loss: 12181.219
Epoch : 7 || Train Loss: 12212.662 || Test Loss: 12163.329
Epoch : 8 || Train Loss: 12168.66 || Test Loss: 12133.063
Epoch : 9 || Train Loss: 12134.602 || Test Loss: 12085.917
Epoch : 10 || Train Loss: 12106.291 || Test Loss: 12072.544
Epoch : 11 || Train Loss: 12081.635 || Test Loss: 12039.811
Epoch : 12 || Train Loss: 12061.085 || Test Loss: 12045.742
Epoch : 13 || Train Loss: 12040.067 || Test Loss: 12000.07
Epoch : 14 || Train Loss: 12020.954 || Test Loss: 11999.054
Epoch : 15 || Train Loss: 12006.845 || Test Loss: 11959.38
Epoch : 16 || Train Loss: 11992.334 || Test Loss: 11961.948
Epoch : 17 || Train Loss: 11967.454 || Test Loss: 119

In [28]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvae_5 = ZINBCSVAE(2000, [1], latent_dim=10, w_dim=10, w_locs = [0.0, 1.0, 2.0, 3.0, 4.0], w_scales = [0.5, 0.5, 0.5, 0.5, 0.5], recon_weight = 20.0, z_kl_weight = 0.2, w_kl_weight = 1.0, adversarial_weight = 10)
csvae_trainer_5 = EpochPyroTrainer(500, csvae_5, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
csvae_trainer_5.train()

Epoch : 1 || Train Loss: 23460.578 || Test Loss: 13322.372
Epoch : 2 || Train Loss: 12878.452 || Test Loss: 12664.131
Epoch : 3 || Train Loss: 12552.06 || Test Loss: 12453.77
Epoch : 4 || Train Loss: 12413.102 || Test Loss: 12349.926
Epoch : 5 || Train Loss: 12324.21 || Test Loss: 12271.377
Epoch : 6 || Train Loss: 12252.875 || Test Loss: 12188.318
Epoch : 7 || Train Loss: 12204.138 || Test Loss: 12145.647
Epoch : 8 || Train Loss: 12156.569 || Test Loss: 12097.763
Epoch : 9 || Train Loss: 12122.319 || Test Loss: 12071.29
Epoch : 10 || Train Loss: 12093.023 || Test Loss: 12054.422
Epoch : 11 || Train Loss: 12067.516 || Test Loss: 12024.003
Epoch : 12 || Train Loss: 12046.626 || Test Loss: 12023.897
Epoch : 13 || Train Loss: 12025.094 || Test Loss: 11992.746
Epoch : 14 || Train Loss: 12005.772 || Test Loss: 11975.477
Epoch : 15 || Train Loss: 11989.134 || Test Loss: 11939.565
Epoch : 16 || Train Loss: 11977.101 || Test Loss: 11942.758
Epoch : 17 || Train Loss: 11951.88 || Test Loss: 1192

In [29]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvae_8 = ZINBCSVAE(2000, [1], latent_dim=10, w_dim=10, w_locs = [0.0, 1.0, 2.0, 3.0, 4.0], w_scales = [0.8, 0.8, 0.8, 0.8, 0.8], recon_weight = 20.0, z_kl_weight = 0.2, w_kl_weight = 1.0, adversarial_weight = 10)
csvae_trainer_8 = EpochPyroTrainer(500, csvae_8, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
csvae_trainer_8.train()

Epoch : 1 || Train Loss: 23393.232 || Test Loss: 13290.359
Epoch : 2 || Train Loss: 12854.382 || Test Loss: 12651.273
Epoch : 3 || Train Loss: 12546.93 || Test Loss: 12442.753
Epoch : 4 || Train Loss: 12416.418 || Test Loss: 12364.024
Epoch : 5 || Train Loss: 12327.196 || Test Loss: 12275.021
Epoch : 6 || Train Loss: 12255.33 || Test Loss: 12185.415
Epoch : 7 || Train Loss: 12207.645 || Test Loss: 12144.427
Epoch : 8 || Train Loss: 12161.576 || Test Loss: 12105.914
Epoch : 9 || Train Loss: 12127.621 || Test Loss: 12086.089
Epoch : 10 || Train Loss: 12101.32 || Test Loss: 12054.111
Epoch : 11 || Train Loss: 12072.072 || Test Loss: 12030.229
Epoch : 12 || Train Loss: 12051.944 || Test Loss: 12055.086
Epoch : 13 || Train Loss: 12024.94 || Test Loss: 11987.053
Epoch : 14 || Train Loss: 12008.093 || Test Loss: 11984.974
Epoch : 15 || Train Loss: 11991.607 || Test Loss: 11944.132
Epoch : 16 || Train Loss: 11977.466 || Test Loss: 11928.827
Epoch : 17 || Train Loss: 11951.815 || Test Loss: 119

In [31]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvae_10 = ZINBCSVAE(2000, [1], latent_dim=10, w_dim=10, w_locs = [0.0, 1.0, 2.0, 3.0, 4.0], w_scales = [1.0, 1.0, 1.0, 1.0, 1.0], recon_weight = 20.0, z_kl_weight = 0.2, w_kl_weight = 1.0, adversarial_weight = 10)
csvae_trainer_10 = EpochPyroTrainer(500, csvae_10, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
csvae_trainer_10.train()

Epoch : 1 || Train Loss: 23381.986 || Test Loss: 13269.493
Epoch : 2 || Train Loss: 12841.364 || Test Loss: 12657.565
Epoch : 3 || Train Loss: 12537.864 || Test Loss: 12435.533
Epoch : 4 || Train Loss: 12407.708 || Test Loss: 12356.454
Epoch : 5 || Train Loss: 12319.19 || Test Loss: 12267.942
Epoch : 6 || Train Loss: 12247.241 || Test Loss: 12187.922
Epoch : 7 || Train Loss: 12202.176 || Test Loss: 12140.052
Epoch : 8 || Train Loss: 12155.947 || Test Loss: 12107.2
Epoch : 9 || Train Loss: 12125.0 || Test Loss: 12075.02
Epoch : 10 || Train Loss: 12096.901 || Test Loss: 12073.409
Epoch : 11 || Train Loss: 12072.225 || Test Loss: 12024.259
Epoch : 12 || Train Loss: 12052.817 || Test Loss: 12030.602
Epoch : 13 || Train Loss: 12026.419 || Test Loss: 11985.261
Epoch : 14 || Train Loss: 12007.642 || Test Loss: 11978.142
Epoch : 15 || Train Loss: 11991.535 || Test Loss: 11942.166
Epoch : 16 || Train Loss: 11977.385 || Test Loss: 11939.277
Epoch : 17 || Train Loss: 11952.378 || Test Loss: 11930

In [34]:
# csvae_1.save("params/hp_tune/csvae_liverfeed_1")
# csvae_2.save("params/hp_tune/csvae_liverfeed_2")
csvae_5.save("params/hp_tune/csvae_liverfeed_5")
csvae_8.save("params/hp_tune/csvae_liverfeed_8")
csvae_10.save("params/hp_tune/csvae_liverfeed_10")

In [24]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvae_1_adv1 = ZINBCSVAE(2000, [1], latent_dim=10, w_dim=10, w_locs = [0.0, 1.0, 2.0, 3.0, 4.0], w_scales = [0.1, 0.1, 0.1, 0.1, 0.1], recon_weight = 20.0, z_kl_weight = 0.2, w_kl_weight = 1.0, adversarial_weight = 1)
csvae_trainer_1_adv1 = EpochPyroTrainer(500, csvae_1, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
csvae_trainer_1_adv1.train()

Epoch : 1 || Train Loss: 11664.968 || Test Loss: 11614.309
Epoch : 2 || Train Loss: 11634.221 || Test Loss: 11587.317
Epoch : 3 || Train Loss: 11607.239 || Test Loss: 11577.683
Epoch : 4 || Train Loss: 11592.748 || Test Loss: 11545.154
Epoch : 5 || Train Loss: 11580.25 || Test Loss: 11541.582
Epoch : 6 || Train Loss: 11562.111 || Test Loss: 11512.708
Epoch : 7 || Train Loss: 11558.862 || Test Loss: 11504.74
Epoch : 8 || Train Loss: 11544.851 || Test Loss: 11493.74
Epoch : 9 || Train Loss: 11531.72 || Test Loss: 11484.866
Epoch : 10 || Train Loss: 11525.14 || Test Loss: 11507.468
Epoch : 11 || Train Loss: 11518.271 || Test Loss: 11466.105
Epoch : 12 || Train Loss: 11509.84 || Test Loss: 11460.997
Epoch : 13 || Train Loss: 11500.063 || Test Loss: 11458.638
Epoch : 14 || Train Loss: 11497.813 || Test Loss: 11457.104
Epoch : 15 || Train Loss: 11494.721 || Test Loss: 11431.029
Epoch : 16 || Train Loss: 11491.854 || Test Loss: 11450.57
Epoch : 17 || Train Loss: 11479.245 || Test Loss: 11429.

In [25]:
get_latents(original_latents, csvae_trainer_1_adv1, 'csvae_1_adv1')

In [26]:
print_all_MI(original_mi_dict, 'csvae_1_adv1', original_latents['csvae_1_adv1'])

C:\Users\anany\AppData\Local\Temp\ipykernel_11856\1545709107.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  var_tensor = torch.tensor(var, dtype=torch.float32)
100%|███████████████████████████████████████████████████████████████████████████| 18715/18715 [01:01<00:00, 303.78it/s]


In [35]:
get_latents(original_latents, csvae_trainer_5, 'csvae_5')
get_latents(original_latents, csvae_trainer_8, 'csvae_8')
get_latents(original_latents, csvae_trainer_10, 'csvae_10')

In [36]:
print_all_MI(original_mi_dict, 'csvae_5', original_latents['csvae_5'])
print_all_MI(original_mi_dict, 'csvae_8', original_latents['csvae_8'])
print_all_MI(original_mi_dict, 'csvae_10', original_latents['csvae_10'])

C:\Users\anany\AppData\Local\Temp\ipykernel_11856\1545709107.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  var_tensor = torch.tensor(var, dtype=torch.float32)
100%|███████████████████████████████████████████████████████████████████████████| 18715/18715 [00:58<00:00, 318.42it/s]


In [14]:
def get_latents(latents, model, name):
    preds = model.get_variables('test')
    z_s = preds['z'][0].cpu()
    w_s = preds['w'][0].cpu()
    latents[name] = {"w": w_s, "z": z_s}

In [17]:
original_latents = {}
get_latents(original_latents, csvae_trainer_1, 'csvae_1')
get_latents(original_latents, csvae_trainer_2, 'csvae_2')

In [18]:
sys.path.append("./FlatlandAndBeyond/GeometriclyAgnosticEvaluation/src/")
from agnostic_metrics import *

In [19]:
def get_normalized_distance(var):
    var_tensor = torch.tensor(var, dtype=torch.float32)
    euclidean_var = get_pairwise_euclidean_distance(var_tensor)
    normalized_var = normalise_distances(euclidean_var)
    return normalized_var

In [20]:
time_labels = anndata.obs['time'].cat.codes.values
# T00 -> 0, T06 -> 1, T12 -> 2, T18 -> 3
time_labels = torch.tensor(time_labels, dtype=torch.long)

# zone_labels = anndata.obs['zone'].cat.codes.values
# zone_labels = torch.tensor(zone_labels, dtype=torch.long)

# get_normalized_distance(keyval['z'])
zone_distance = get_normalized_distance(anndata.obs['zone'].to_numpy().astype(float).reshape(-1,1))

In [21]:
def print_all_MI(mi_dict, model, keyval):
    z_norm = get_normalized_distance(keyval['z'])
    w_norm = get_normalized_distance(keyval['w'])
    mi_dict[model] = {}
    calculate_MI(mi_dict, model, z_norm, w_norm)

def calculate_MI(mi_dict, model, z, w):
    time_distance = compute_categorical_distance(time_labels.cpu().numpy(), discrete_dist=1)
    # zone_distance = compute_categorical_distance(zone_labels.cpu().numpy(), discrete_dist=1)
    distances = np.concatenate(
        [z.cpu().numpy()[None, :, :], 
         w.cpu().numpy()[None, :, :], 
         time_distance[None, :, :], 
         zone_distance[None, :, :]],
        axis=0)
    mi_wz = cmi([1], [0], [], 5, distances, precomputed=True)
    mi_ztime = cmi([0], [2], [], 5, distances, precomputed=True)
    mi_zzone = cmi([0], [3], [], 5, distances, precomputed=True)
    mi_wtime = cmi([1], [2], [], 5, distances, precomputed=True)
    mi_wzone = cmi([1], [3], [], 5, distances, precomputed=True)
    mi_dict[model]["I(w,time)"] = mi_wtime
    mi_dict[model]["I(w,zone)"] = mi_wzone
    mi_dict[model]["I(z,time)"] = mi_ztime
    mi_dict[model]["I(z,zone)"] = mi_zzone
    mi_dict[model]["I(w,z)"] = mi_wz

In [22]:
original_mi_dict = {}
for model, vals in original_latents.items():
    print_all_MI(original_mi_dict, model, vals)

C:\Users\anany\AppData\Local\Temp\ipykernel_11856\1545709107.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  var_tensor = torch.tensor(var, dtype=torch.float32)
100%|███████████████████████████████████████████████████████████████████████████| 18715/18715 [00:56<00:00, 329.94it/s]


In [37]:
df = pd.DataFrame(original_mi_dict)
print(df.transpose())

              I(w,time)  I(w,zone)  I(z,time)  I(z,zone)    I(w,z)
csvae_1        1.493864   0.221675   1.428381   0.879209  1.426426
csvae_2        1.490770   0.359576   1.464603   0.893339  1.648610
csvae_1_adv1   1.493968   0.233206   1.425871   0.869483  1.424698
csvae_5        1.398316   0.911580   1.472846   0.552114  2.145926
csvae_8        1.291382   0.871217   1.483933   0.454646  2.119844
csvae_10       1.269006   0.828753   1.481424   0.466337  2.178017
